# Train Classifier on BirdNET Embeddings

This notebook trains a shallow classifier on pre-computed BirdNET embeddings using the train/test/validation splits from YOLO.

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
import matplotlib.pyplot as plt
import seaborn as sns

## Load BirdNET Embeddings

In [ ]:
# Load BirdNET embeddings (one embedding per individual call clip)
embeddings_path = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/birds_embeddings_birdnet_padded_3s.csv'
embeddings_df = pd.read_csv(embeddings_path)

# Extract trial ID from the file path to match with YOLO splits
# Example: .../SL07_Trial_1_trim_37.44715902_37.57169857_D.WAV -> SL07_Trial_1_trim
def extract_trial_id(filepath):
    filename = filepath.split('/')[-1]  # Get filename
    # Remove the timestamp and call type parts: file_id_begin_end_calltype.WAV
    parts = filename.rsplit('_', 3)  # Split from right to remove begin_end_calltype
    return parts[0]  # Return just the trial ID

embeddings_df['trial_id'] = embeddings_df['file'].apply(extract_trial_id)

# Use the calltype column that already exists in the CSV
print(f"Loaded embeddings: {embeddings_df.shape}")
print(f"Call types found: {embeddings_df['calltype'].unique()}")
print(f"Sample trial IDs: {embeddings_df['trial_id'].unique()[:5]}")
embeddings_df.head()

Loaded embeddings: (6047, 1024)
First few rows of embeddings index:
Index([    ('/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL07_Trial_1_trim_37.44715902_37.57169857_D.WAV', 0.0, 3.0),
          ('/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL17a_Trial_1_trim_41.11663135_41.18140915_F.WAV', 0.0, 3.0),
           ('/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL10_Trial_1_trim_28.58731405_28.69749079_F.WAV', 0.0, 3.0),
       ('/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL10_Trial_1_trim_47.70134954_48.36720022_Cheer.WAV', 0.0, 3.0),
       ('/home/Shelby/blackbird_calls/Experiments/Detect

/tmp/ipykernel_533657/3104284505.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  embeddings_df['clip_index'] = list(zip(embeddings_df['file'], embeddings_df['start_time'], embeddings_df['end_time']))


,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
clip_index,,,,,,,,,,,,,,,,,,,,,
"(/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL07_Trial_1_trim_37.44715902_37.57169857_D.WAV, 0.0, 3.0)",0.067894,0.344097,0.135184,0.049056,0.659216,1.029421,0.000000,0.302469,0.341832,0.015681,...,0.000000,0.000000,0.283043,0.353124,0.850524,0.342322,0.000000,1.061636,0.762078,0.485685
"(/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL17a_Trial_1_trim_41.11663135_41.18140915_F.WAV, 0.0, 3.0)",0.027922,0.142469,0.005187,0.067271,0.845247,0.754935,0.000000,0.503155,0.199898,0.000000,...,0.000000,0.000000,0.565755,0.272597,0.313492,0.231603,0.067483,1.466044,1.240883,0.785682
"(/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL10_Trial_1_trim_28.58731405_28.69749079_F.WAV, 0.0, 3.0)",0.063137,0.392312,0.081558,0.013896,0.519530,0.799999,0.007829,0.299188,0.276275,0.115706,...,0.007743,0.083630,0.350261,0.338610,0.525507,0.445224,0.000000,0.847136,0.351975,0.716970
"(/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL10_Trial_1_trim_47.70134954_48.36720022_Cheer.WAV, 0.0, 3.0)",0.247558,0.025925,0.128917,0.579505,0.000000,0.203117,0.222648,0.000000,1.270295,0.099739,...,0.250644,0.000000,0.057046,0.103796,1.025722,0.991909,0.000000,0.515161,0.025441,1.636110
"(/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/AZ13_Trial_3_trim_21.29153733_21.35859514_Check.WAV, 0.0, 3.0)",0.023261,0.383073,0.003367,0.000000,0.256864,0.177266,0.293753,0.069218,0.074445,0.150925,...,0.000000,0.000597,1.142428,0.817335,0.477512,0.280269,0.000000,2.258079,0.730854,0.299776


## Load Train/Test/Validation Splits

In [ ]:
# Load YOLO split files to get trial IDs for train/test/val
import os

split_dir = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/YOLO/dataset_split/'

def extract_trial_ids_from_yolo(txt_file):
    """Extract unique trial identifiers from YOLO split file"""
    with open(txt_file, 'r') as f:
        paths = [line.strip() for line in f]
    
    trial_ids = set()
    for path in paths:
        filename = os.path.basename(path)
        # Remove _spec_XX-XX.png to get trial ID
        trial_id = filename.rsplit('_spec_', 1)[0]
        trial_ids.add(trial_id)
    
    return trial_ids

# Get trial IDs for each split
test_trials = extract_trial_ids_from_yolo(os.path.join(split_dir, 'test.txt'))
val_trials = extract_trial_ids_from_yolo(os.path.join(split_dir, 'validate.txt'))
train_trials = extract_trial_ids_from_yolo(os.path.join(split_dir, 'train.txt'))

print(f"Test trials ({len(test_trials)}): {sorted(test_trials)[:5]}...")
print(f"Val trials ({len(val_trials)}): {sorted(val_trials)[:5]}...")
print(f"Train trials ({len(train_trials)}): {sorted(list(train_trials)[:5])}...")

Training labels: (1560, 19)
Validation labels: (320, 19)
Test labels: (301, 19)

Call types: ['Cheer', 'A', 'Check', 'B', 'K', 'C', 'Chits', 'M', 'Growl', 'D', 'J', 'O', 'I', 'F', 'G', 'H', 'E', 'N', 'L']

Sample train_labels index:
MultiIndex([('/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', ...),
            ('/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', ...),
            ('/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', ...)],
           names=['file', 'start_time', 'end_time'])


## Split Data by Trial IDs

In [ ]:
# Split embeddings by trial ID
train_mask = embeddings_df['trial_id'].isin(train_trials)
val_mask = embeddings_df['trial_id'].isin(val_trials)
test_mask = embeddings_df['trial_id'].isin(test_trials)

train_df = embeddings_df[train_mask]
val_df = embeddings_df[val_mask]
test_df = embeddings_df[test_mask]

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} / {len(embeddings_df)}")

# Prepare X (embeddings) and y (labels)
# Embedding columns are numbered 0-1023
embedding_cols = [str(i) for i in range(1024)]

X_train = train_df[embedding_cols].values
y_train = train_df['calltype'].values

X_val = val_df[embedding_cols].values
y_val = val_df['calltype'].values

X_test = test_df[embedding_cols].values
y_test = test_df['calltype'].values

print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Training samples with embeddings: 0 / 1560
Validation samples with embeddings: 0 / 320
Test samples with embeddings: 0 / 301

X_train shape: (0, 1024), y_train shape: (0, 19)
X_val shape: (0, 1024), y_val shape: (0, 19)
X_test shape: (0, 1024), y_test shape: (0, 19)


In [8]:
# Debug: Check the format of indices
print("Sample embeddings indices:")
print(embeddings.index[:5].tolist())
print("\nSample train_labels indices:")
print(train_labels.index[:5].tolist())
print("\nAre they the same type?")
print(f"Embeddings index type: {type(embeddings.index[0])}")
print(f"Labels index type: {type(train_labels.index[0])}")

Sample embeddings indices:
['D', 'F', 'F', 'Cheer', 'Check']

Sample train_labels indices:
['/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', '/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', '/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', '/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV', '/mnt/class_data/Shelby/One_Minute_Audio/AZ02_Trial_2_trim.WAV']

Are they the same type?
Embeddings index type: <class 'str'>
Labels index type: <class 'str'>


In [9]:
# Check the structure of the embeddings file more carefully
print("Embeddings DataFrame info:")
print(f"Shape: {embeddings.shape}")
print(f"\nFirst few column names:")
print(embeddings.columns[:10].tolist())
print(f"\nFirst few index values:")
print(embeddings.index[:10].tolist())
print(f"\nDoes this look like it has audio file paths as index? Or call types?")

Embeddings DataFrame info:
Shape: (6047, 1027)

First few column names:
['file', 'start_time', 'end_time', '0', '1', '2', '3', '4', '5', '6']

First few index values:
['D', 'F', 'F', 'Cheer', 'Check', 'Cheer', 'Check', 'I', 'Cheer', 'Cheer']

Does this look like it has audio file paths as index? Or call types?


## Check Call Type Distribution

In [ ]:
# Check call type distribution in each split
from collections import Counter

print("Call type distribution in training set:")
train_counts = Counter(y_train)
for call_type, count in sorted(train_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {call_type}: {count}")

print("\nCall type distribution in validation set:")
val_counts = Counter(y_val)
for call_type, count in sorted(val_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {call_type}: {count}")

print("\nCall type distribution in test set:")
test_counts = Counter(y_test)
for call_type, count in sorted(test_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {call_type}: {count}")

## Train Random Forest Classifier

Single-label classification: each call embedding maps to one call type.

In [ ]:
# Train a Random Forest classifier for single-label classification
print("Training Random Forest classifier...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train the model
model.fit(X_train, y_train)
print("Training complete!")

## Evaluate on Validation Set

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val)

# Get all unique call types
call_types = sorted(set(y_train) | set(y_val) | set(y_test))

# Calculate per-class metrics
print("Validation Set Performance:")
print("=" * 80)
print(classification_report(y_val, y_val_pred, labels=call_types, zero_division=0))

In [ ]:
# Calculate overall validation accuracy
from sklearn.metrics import accuracy_score

val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"\nOverall Validation Accuracy: {val_accuracy:.4f}")

## Evaluate on Test Set

In [ ]:
# Predict on test set
y_test_pred = model.predict(X_test)

# Calculate per-class metrics
print("Test Set Performance:")
print("=" * 80)
print(classification_report(y_test, y_test_pred, labels=call_types, zero_division=0))

In [ ]:
# Calculate overall test accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nOverall Test Accuracy: {test_accuracy:.4f}")

## Visualize Confusion Matrices

In [ ]:
# Plot confusion matrix for all call types
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred, labels=call_types)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=call_types, yticklabels=call_types, cmap='Blues')
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Call Type')
plt.xlabel('Predicted Call Type')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [2]:
# Train Classifier on BirdNET Embeddings

#This notebook trains a shallow classifier on pre-computed BirdNET embeddings using the train/test/validation splits from YOLO.

In [ ]:
# Plot confusion matrices for each call type
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, call_type in enumerate(call_types):
    if i < len(axes):
        cm = confusion_matrix(y_test[:, i], y_test_pred[:, i])
        sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], 
                   xticklabels=['Absent', 'Present'],
                   yticklabels=['Absent', 'Present'],
                   cmap='Blues')
        axes[i].set_title(f'{call_type}')
        axes[i].set_ylabel('True')
        axes[i].set_xlabel('Predicted')

# Hide extra subplots if there are fewer call types than subplots
for i in range(len(call_types), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Visualize Confusion Matrices

In [ ]:
# Calculate overall test metrics
print("\nOverall Test Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision (macro): {precision_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (macro): {recall_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (macro): {f1_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")

In [ ]:
# Predict on test set
y_test_pred = model.predict(X_test)

# Calculate per-class metrics
print("Test Set Performance (per call type):")
print("=" * 80)
for i, call_type in enumerate(call_types):
    print(f"\n{call_type}:")
    print(classification_report(y_test[:, i], y_test_pred[:, i], 
                                target_names=['Absent', 'Present'],
                                zero_division=0))

## Evaluate on Test Set

In [ ]:
# Calculate overall accuracy metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\nOverall Validation Metrics:")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision (macro): {precision_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (macro): {recall_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (macro): {f1_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val)

# Calculate per-class metrics
print("Validation Set Performance (per call type):")
print("=" * 80)
for i, call_type in enumerate(call_types):
    print(f"\n{call_type}:")
    print(classification_report(y_val[:, i], y_val_pred[:, i], 
                                target_names=['Absent', 'Present'],
                                zero_division=0))

## Evaluate on Validation Set

In [ ]:
# Train a Random Forest classifier for multi-label classification
print("Training Random Forest classifier...")
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Wrap in MultiOutputClassifier for multi-label
model = MultiOutputClassifier(rf_classifier, n_jobs=-1)

# Train the model
model.fit(X_train, y_train)
print("Training complete!")

## Train Random Forest Classifier

Using a Random Forest wrapped in MultiOutputClassifier for multi-label classification.

In [ ]:
# Check class distribution in each split
call_types = train_labels.columns

print("Class distribution in training set:")
train_dist = pd.DataFrame({
    'Call Type': call_types,
    'Count': y_train.sum(axis=0)
}).sort_values('Count', ascending=False)
print(train_dist)

print("\nClass distribution in validation set:")
val_dist = pd.DataFrame({
    'Call Type': call_types,
    'Count': y_val.sum(axis=0)
}).sort_values('Count', ascending=False)
print(val_dist)

print("\nClass distribution in test set:")
test_dist = pd.DataFrame({
    'Call Type': call_types,
    'Count': y_test.sum(axis=0)
}).sort_values('Count', ascending=False)
print(test_dist)

## Check Class Distribution

In [ ]:
# Find common indices between embeddings and labels
train_common = train_labels.index.intersection(embeddings.index)
val_common = val_labels.index.intersection(embeddings.index)
test_common = test_labels.index.intersection(embeddings.index)

print(f"Training samples with embeddings: {len(train_common)} / {len(train_labels)}")
print(f"Validation samples with embeddings: {len(val_common)} / {len(val_labels)}")
print(f"Test samples with embeddings: {len(test_common)} / {len(test_labels)}")

# Prepare training data
X_train = embeddings.loc[train_common].values
y_train = train_labels.loc[train_common].values

# Prepare validation data
X_val = embeddings.loc[val_common].values
y_val = val_labels.loc[val_common].values

# Prepare test data
X_test = embeddings.loc[test_common].values
y_test = test_labels.loc[test_common].values

print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## Merge Embeddings with Labels

In [ ]:
# Load train/test/validation splits with labels
train_labels = pd.read_csv('./annotated_data/train_set.csv', index_col=0)
val_labels = pd.read_csv('./annotated_data/val_set.csv', index_col=0)
test_labels = pd.read_csv('./annotated_data/test_set.csv', index_col=0)

print(f"Training labels: {train_labels.shape}")
print(f"Validation labels: {val_labels.shape}")
print(f"Test labels: {test_labels.shape}")
print(f"\nCall types: {list(train_labels.columns)}")

## Load Train/Test/Validation Splits

In [ ]:
# Load BirdNET embeddings
embeddings_path = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/birds_embeddings_birdnet_padded_3s.csv'
embeddings = pd.read_csv(embeddings_path, index_col=0)

print(f"Loaded embeddings: {embeddings.shape}")
print(f"First few rows of embeddings index:")
print(embeddings.index[:5])
embeddings.head()

## Load BirdNET Embeddings

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
import matplotlib.pyplot as plt
import seaborn as sns